In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/pssd.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Which SSRIs cause the worst PSSD (Post-SSRI Sexual Dysfunction), and what predicts a more severe case?"

*Focus: Harm and causation. Recovery analysis is covered separately in NB6.*

## Abstract

This analysis examines which serotonergic antidepressants are most frequently implicated in Post-SSRI Sexual Dysfunction (PSSD) and what patient-level factors predict more severe outcomes. Drawing on 902 treatment reports from 220 unique reporters in r/PSSD (March 12 -- April 11, 2026), we find that **sertraline (Zoloft) is the most frequently reported causative SSRI** with the highest negative-outcome rate (94.9% of 39 users classified negative) and zero positive reports. Paroxetine, duloxetine, and venlafaxine show 100% negative rates but with smaller samples (n=5--7). Key severity predictors include symptom domain breadth -- duloxetine and citalopram users report multi-domain dysfunction (sexual, emotional, cognitive) at far higher rates than sertraline users. Polypharmacy does not significantly worsen outcomes beyond the floor effect already present. Vortioxetine, marketed as sexually-sparing, shows 87.5% negative with zero positive reports (n=8), contradicting its clinical positioning.

## Data Overview

Data covers: **2026-03-12 to 2026-04-11** (1 month). The r/PSSD community is a patient-led forum for people experiencing persistent sexual, emotional, and cognitive dysfunction after discontinuing serotonergic antidepressants.

**Important context:** In this community, SSRIs are the *cause* of the condition, not treatments being evaluated for efficacy. Negative sentiment toward sertraline in r/PSSD means "sertraline caused my PSSD" -- not "sertraline failed to treat my PSSD." This reverses the usual interpretation: a high negative rate for a drug reflects how frequently it is blamed for causing harm, not treatment failure.

In [ ]:

total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM posts", conn).iloc[0,0]
total_posts = pd.read_sql("SELECT COUNT(*) FROM posts", conn).iloc[0,0]
total_reports = pd.read_sql("SELECT COUNT(*) FROM treatment_reports", conn).iloc[0,0]
unique_reporters = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM treatment_reports", conn).iloc[0,0]
date_range = pd.read_sql("SELECT MIN(date(post_date, 'unixepoch')) as min_d, MAX(date(post_date, 'unixepoch')) as max_d FROM posts", conn)

summary_html = f"""
<table style='border-collapse:collapse; font-size:14px;'>
<tr><td style='padding:4px 12px; font-weight:bold;'>Community members</td><td style='padding:4px 12px;'>{total_users:,}</td></tr>
<tr><td style='padding:4px 12px; font-weight:bold;'>Total posts</td><td style='padding:4px 12px;'>{total_posts:,}</td></tr>
<tr><td style='padding:4px 12px; font-weight:bold;'>Treatment reports</td><td style='padding:4px 12px;'>{total_reports:,}</td></tr>
<tr><td style='padding:4px 12px; font-weight:bold;'>Unique reporters</td><td style='padding:4px 12px;'>{unique_reporters:,}</td></tr>
<tr><td style='padding:4px 12px; font-weight:bold;'>Date range</td><td style='padding:4px 12px;'>{date_range.iloc[0,0]} to {date_range.iloc[0,1]}</td></tr>
</table>
"""
display(HTML(summary_html))


## Which SSRIs Are Most Frequently Blamed for PSSD?

Before examining severity, we need to establish which drugs dominate the conversation. The following analysis merges brand and generic names (lexapro/escitalopram, prozac/fluoxetine) and uses user-level aggregation so each person counts once per drug regardless of how many posts they made.

**Methodology:** Sentiment is aggregated at the user level: each user's reports for a given drug are averaged, then classified as negative (avg < -0.3), positive (avg > 0.7), or mixed/neutral. Wilson score confidence intervals account for small sample sizes. Generic category terms ("ssri", "antidepressant", "snri") are excluded -- only specific drug names are analyzed.

In [ ]:

MERGE_MAP = {'lexapro': 'escitalopram', 'prozac': 'fluoxetine'}
SSRI_DRUGS = ['sertraline', 'fluoxetine', 'paroxetine', 'escitalopram', 'citalopram',
              'vortioxetine', 'duloxetine', 'venlafaxine', 'prozac', 'lexapro']

user_drug = pd.read_sql(
    "SELECT tr.user_id, t.canonical_name, "
    "AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    "WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_score, "
    "COUNT(*) as n_reports "
    "FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id "
    "WHERE t.canonical_name IN (" + ",".join(["?"] * len(SSRI_DRUGS)) + ") "
    "GROUP BY tr.user_id, t.canonical_name",
    conn, params=SSRI_DRUGS)

user_drug['drug_merged'] = user_drug['canonical_name'].map(MERGE_MAP).fillna(user_drug['canonical_name'])
user_drug_merged = user_drug.groupby(['user_id', 'drug_merged']).agg(
    avg_score=('avg_score', 'mean'), n_reports=('n_reports', 'sum')).reset_index()
user_drug_merged['outcome'] = user_drug_merged['avg_score'].apply(classify_outcome)

drug_stats = []
for drug, grp in user_drug_merged.groupby('drug_merged'):
    n = len(grp)
    n_neg = (grp['outcome'] == 'negative').sum()
    n_pos = (grp['outcome'] == 'positive').sum()
    n_mix = (grp['outcome'] == 'mixed/neutral').sum()
    neg_rate = n_neg / n
    pos_rate = n_pos / n
    neg_lo, neg_hi = wilson_ci(n_neg, n)
    pos_lo, pos_hi = wilson_ci(n_pos, n)
    drug_stats.append({'drug': drug, 'users': n, 'neg': n_neg, 'pos': n_pos, 'mix': n_mix,
        'neg_rate': neg_rate, 'pos_rate': pos_rate,
        'neg_ci_lo': neg_lo, 'neg_ci_hi': neg_hi, 'pos_ci_lo': pos_lo, 'pos_ci_hi': pos_hi})
drug_df = pd.DataFrame(drug_stats).sort_values('users', ascending=False)

display_df = drug_df[['drug', 'users', 'neg', 'pos', 'mix', 'neg_rate', 'pos_rate']].copy()
display_df.columns = ['Drug', 'Users', 'Negative', 'Positive', 'Mixed/Neutral', 'Neg Rate', 'Pos Rate']
display_df['Drug'] = display_df['Drug'].str.title()
display_df['Neg Rate'] = display_df['Neg Rate'].apply(lambda x: f"{x:.1%}")
display_df['Pos Rate'] = display_df['Pos Rate'].apply(lambda x: f"{x:.1%}")
display(HTML("<h4>SSRI/SNRI Causation Reports (User-Level, Merged Brand/Generic)</h4>"))
display(HTML(display_df.to_html(index=False)))


**Key observation:** Every serotonergic antidepressant in this community carries a negative-outcome rate above 70%. This is expected -- people join r/PSSD *because* an SSRI harmed them. The variation within this near-universally negative space is what matters: sertraline has 39 users and 94.9% negative, while escitalopram/lexapro (merged) shows a notably lower negative rate with some positive reports. The question is whether these differences reflect genuine differential harm or sampling variation.

In [ ]:

plot_df = drug_df[drug_df['users'] >= 4].sort_values('neg_rate', ascending=True).copy()
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(plot_df))

ax.errorbar(plot_df['neg_rate'].values * 100, list(y_pos),
    xerr=[(plot_df['neg_rate'].values - plot_df['neg_ci_lo'].values) * 100,
          (plot_df['neg_ci_hi'].values - plot_df['neg_rate'].values) * 100],
    fmt='none', ecolor='#7f8c8d', elinewidth=2, capsize=4, zorder=4)

for i, (_, row) in enumerate(plot_df.iterrows()):
    color = '#e74c3c' if row['neg_rate'] >= 0.9 else '#e67e22' if row['neg_rate'] >= 0.8 else '#f1c40f'
    ax.plot(row['neg_rate'] * 100, i, 'o', color=color, markersize=10, zorder=6)

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{row['drug'].title()} (n={row['users']})" for _, row in plot_df.iterrows()], fontsize=11)
ax.set_xlabel('Negative Outcome Rate (%)', fontsize=12)
ax.set_title('PSSD Causation: Negative Outcome Rate by Drug\n(Wilson 95% CI)', fontsize=13, fontweight='bold')
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='>=90% negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e67e22', markersize=10, label='80-89% negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f1c40f', markersize=10, label='<80% negative')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.set_xlim(30, 105)
fig.tight_layout()
plt.show()


**What this chart shows:** All SSRIs cluster above 70% negative, but the drugs with the tightest confidence intervals -- sertraline (n=39) and escitalopram/lexapro (n=29 merged) -- tell the most reliable story. Sertraline's CI sits entirely above 80%, confirming its reputation as the most-reported PSSD causative agent. Paroxetine, duloxetine, and venlafaxine hit 100% negative but with wide CIs reflecting small samples (n=4--7) -- we cannot reliably distinguish them from the 80--90% range.

In [ ]:

import math

sert_users = user_drug_merged[user_drug_merged['drug_merged'] == 'sertraline']
esci_users = user_drug_merged[user_drug_merged['drug_merged'] == 'escitalopram']

sert_neg = (sert_users['outcome'] == 'negative').sum()
sert_n = len(sert_users)
esci_neg = (esci_users['outcome'] == 'negative').sum()
esci_n = len(esci_users)

table_fe = [[sert_neg, sert_n - sert_neg], [esci_neg, esci_n - esci_neg]]
odds_ratio_val, p_fisher = fisher_exact(table_fe)

p1 = sert_neg / sert_n
p2 = esci_neg / esci_n
cohens_h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

result_html = f"""
<h4>Sertraline vs Escitalopram (merged with Lexapro) -- Fisher's Exact Test</h4>
<table style='border-collapse:collapse; font-size:14px; margin:10px 0;'>
<tr style='background:#f8f9fa;'><th style='padding:6px 12px; text-align:left;'>Metric</th><th style='padding:6px 12px;'>Value</th></tr>
<tr><td style='padding:6px 12px;'>Sertraline negative rate</td><td style='padding:6px 12px;'>{p1:.1%} ({sert_neg}/{sert_n})</td></tr>
<tr><td style='padding:6px 12px;'>Escitalopram negative rate</td><td style='padding:6px 12px;'>{p2:.1%} ({esci_neg}/{esci_n})</td></tr>
<tr><td style='padding:6px 12px;'>Fisher's exact p-value</td><td style='padding:6px 12px;'>{p_fisher:.4f}</td></tr>
<tr><td style='padding:6px 12px;'>Odds ratio</td><td style='padding:6px 12px;'>{odds_ratio_val:.2f}</td></tr>
<tr><td style='padding:6px 12px;'>Cohen's h (effect size)</td><td style='padding:6px 12px;'>{cohens_h:.3f}</td></tr>
</table>
"""
display(HTML(result_html))

if p_fisher < 0.05:
    size_label = 'large' if abs(cohens_h) > 0.8 else 'medium' if abs(cohens_h) > 0.5 else 'small'
    display(HTML(f"<p><b>Verdict:</b> The difference is statistically significant (p={p_fisher:.4f}). Sertraline users are significantly more likely to report negative outcomes than escitalopram users, with a {size_label} effect size (Cohen's h={cohens_h:.3f}). In plain language: among PSSD sufferers, those who blame sertraline are more uniformly negative than those who blame escitalopram.</p>"))
else:
    display(HTML(f"<p><b>Verdict:</b> Despite the apparent difference ({p1:.1%} vs {p2:.1%}), it does not reach statistical significance (p={p_fisher:.4f}). With escitalopram at n={esci_n}, we lack the power to confidently distinguish these rates.</p>"))


## What Predicts a More Severe Case?

The previous section established which SSRIs are most-reported. But within this community of people who all have PSSD, some cases appear worse than others. We examine three potential severity predictors:

1. **Polypharmacy** -- Did trying multiple SSRIs worsen outcomes?
2. **Symptom domain breadth** -- Do some drugs produce broader dysfunction (sexual + emotional + cognitive)?
3. **Signal strength** -- Do strong-signal reports (explicit causation language) differ from weaker signals?

### Predictor 1: Multiple SSRI Exposure

23 users in this dataset reported trying 2 or more specific SSRIs (average 2.5 drugs). If polypharmacy worsens PSSD, these users should show worse sentiment than single-SSRI users.

In [ ]:

ssri_per_user = user_drug_merged.groupby('user_id').agg(
    n_ssris=('drug_merged', 'nunique'), mean_score=('avg_score', 'mean')).reset_index()
ssri_per_user['group'] = ssri_per_user['n_ssris'].apply(lambda x: 'Multi-SSRI (2+)' if x > 1 else 'Single SSRI')

multi = ssri_per_user[ssri_per_user['group'] == 'Multi-SSRI (2+)']['mean_score']
single = ssri_per_user[ssri_per_user['group'] == 'Single SSRI']['mean_score']

stat_u, p_mw = mannwhitneyu(multi, single, alternative='two-sided')
n1, n2 = len(multi), len(single)
rbc = 1 - (2 * stat_u) / (n1 * n2)

poly_outcomes = ssri_per_user.copy()
poly_outcomes['outcome'] = poly_outcomes['mean_score'].apply(classify_outcome)
outcome_counts = poly_outcomes.groupby(['group', 'outcome']).size().unstack(fill_value=0)
outcome_pcts = outcome_counts.div(outcome_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(outcome_pcts.index))
width = 0.25

for i, outcome in enumerate(['negative', 'mixed/neutral', 'positive']):
    if outcome in outcome_pcts.columns:
        vals = outcome_pcts[outcome].values
        bars = ax.bar(x + i * width, vals, width, label=outcome.title(), color=COLORS.get(outcome, '#95a5a6'))
        for bar, val in zip(bars, vals):
            if val > 3:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                        f'{val:.0f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x + width)
ax.set_xticklabels([f"{g}\n(n={len(ssri_per_user[ssri_per_user['group']==g])})" for g in outcome_pcts.index], fontsize=11)
ax.set_ylabel('Percentage of Users (%)', fontsize=12)
ax.set_title('Outcome Distribution: Single vs Multi-SSRI Users', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_ylim(0, 110)
fig.tight_layout()
plt.show()

poly_html = f"""
<table style='border-collapse:collapse; font-size:14px; margin:10px 0;'>
<tr style='background:#f8f9fa;'><th style='padding:6px 12px;'>Group</th><th style='padding:6px 12px;'>n</th>
<th style='padding:6px 12px;'>Mean Score</th><th style='padding:6px 12px;'>Median</th></tr>
<tr><td style='padding:6px 12px;'>Single SSRI</td><td style='padding:6px 12px;'>{len(single)}</td>
<td style='padding:6px 12px;'>{single.mean():.3f}</td><td style='padding:6px 12px;'>{single.median():.3f}</td></tr>
<tr><td style='padding:6px 12px;'>Multi-SSRI (2+)</td><td style='padding:6px 12px;'>{len(multi)}</td>
<td style='padding:6px 12px;'>{multi.mean():.3f}</td><td style='padding:6px 12px;'>{multi.median():.3f}</td></tr>
<tr><td style='padding:6px 12px;' colspan=2>Mann-Whitney U p-value</td><td style='padding:6px 12px;' colspan=2>{p_mw:.4f}</td></tr>
<tr><td style='padding:6px 12px;' colspan=2>Rank-biserial r</td><td style='padding:6px 12px;' colspan=2>{rbc:.3f}</td></tr>
</table>
"""
display(HTML(poly_html))

if p_mw < 0.05:
    display(HTML(f"<p><b>Verdict:</b> Multi-SSRI users show significantly different sentiment (p={p_mw:.4f}, r={rbc:.3f}).</p>"))
else:
    display(HTML(f"<p><b>Verdict:</b> No statistically significant difference between multi-SSRI and single-SSRI users (p={p_mw:.4f}). Both groups are overwhelmingly negative. Polypharmacy may not worsen outcomes beyond the floor effect -- nearly everyone reports negative outcomes regardless of how many SSRIs they tried.</p>"))


### Predictor 2: Symptom Domain Profiles by Drug

PSSD manifests across multiple domains: sexual dysfunction (genital numbness, libido loss), emotional blunting (anhedonia, emotional numbness), and cognitive impairment (brain fog, memory issues). If certain SSRIs produce broader damage, users who took those drugs should mention more symptom domains. We search post text for domain-specific keywords at the user level (each user counted once per domain).

In [ ]:

SSRI_SPECIFIC = ['sertraline', 'fluoxetine', 'paroxetine', 'escitalopram', 'citalopram',
                 'vortioxetine', 'duloxetine', 'venlafaxine']

domain_queries = {
    'Sexual': "p.body_text LIKE '%libido%' OR p.body_text LIKE '%sexual%' OR p.body_text LIKE '%genital%' OR p.body_text LIKE '%erectile%' OR p.body_text LIKE '%orgasm%'",
    'Emotional': "p.body_text LIKE '%emotional%' OR p.body_text LIKE '%anhedonia%' OR p.body_text LIKE '%numb%' OR p.body_text LIKE '%blunt%'",
    'Cognitive': "p.body_text LIKE '%cognitive%' OR p.body_text LIKE '%brain fog%' OR p.body_text LIKE '%memory%' OR p.body_text LIKE '%concentration%'",
    'Severity Language': "p.body_text LIKE '%severe%' OR p.body_text LIKE '%permanent%' OR p.body_text LIKE '%irreversible%' OR p.body_text LIKE '%worse%'",
}

results_list = []
for drug in SSRI_SPECIFIC:
    drug_users = pd.read_sql(
        "SELECT DISTINCT tr.user_id FROM treatment_reports tr "
        "JOIN treatment t ON tr.drug_id = t.id WHERE t.canonical_name = ?",
        conn, params=[drug])
    if len(drug_users) < 4:
        continue
    user_ids = drug_users['user_id'].tolist()
    placeholders = ','.join(['?' for _ in user_ids])
    row = {'drug': drug, 'users': len(drug_users)}
    for domain, condition in domain_queries.items():
        count_df = pd.read_sql(
            f"SELECT COUNT(DISTINCT p.user_id) as cnt FROM posts p "
            f"WHERE p.user_id IN ({placeholders}) AND ({condition})",
            conn, params=user_ids)
        row[domain] = count_df.iloc[0, 0] / len(drug_users) * 100
    results_list.append(row)

domain_df = pd.DataFrame(results_list).set_index('drug')
n_users_map = domain_df['users']
heatmap_data = domain_df.drop(columns=['users'])
heatmap_data['total'] = heatmap_data.sum(axis=1)
heatmap_data = heatmap_data.sort_values('total', ascending=True).drop(columns=['total'])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, fontsize=11)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels([f"{d.title()} (n={int(n_users_map.loc[d])})" for d in heatmap_data.index], fontsize=11)

for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.iloc[i, j]
        text_color = 'white' if val > 60 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center', fontsize=11, fontweight='bold', color=text_color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('% of Users Mentioning Domain', fontsize=11)
ax.set_title('Symptom Domain Profiles by Causative SSRI\n(% of drug users mentioning each domain)',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()


**What this chart shows:** The sexual domain is reported by 50--100% of users regardless of drug, confirming it as the core feature of PSSD. The differentiating factor is multi-domain spread. Duloxetine and citalopram users show the broadest symptom involvement, with 60%+ mentioning cognitive symptoms -- compared to under 15% for the larger sertraline and fluoxetine cohorts. Paroxetine stands out for the highest sexual-symptom mention rate (85.7%), consistent with its known strong serotonergic binding affinity.

**Caveat:** Small sample sizes for duloxetine (n=5), citalopram (n=5), and venlafaxine (n=4) mean these high rates could reflect particularly active individual posters rather than true drug-specific effects.

### Predictor 3: Signal Strength as a Severity Proxy

Treatment reports are tagged with signal strength (strong, moderate, weak) based on how explicitly the user connects the drug to an outcome. Strong-signal reports use causal language vs. weak signals that merely mention usage. If strong signals correlate with worse outcomes, signal strength may proxy for severity of experienced harm.

In [ ]:

signal_df = pd.read_sql(
    "SELECT tr.user_id, t.canonical_name, tr.signal_strength, tr.sentiment, "
    "CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    "WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score "
    "FROM treatment_reports tr JOIN treatment t ON tr.drug_id = t.id "
    "WHERE t.canonical_name IN (" + ",".join(["?"] * len(SSRI_DRUGS)) + ")",
    conn, params=SSRI_DRUGS)

signal_df['drug_merged'] = signal_df['canonical_name'].map(MERGE_MAP).fillna(signal_df['canonical_name'])

sig_dist = signal_df.groupby('signal_strength')['score'].agg(['count', 'mean', 'median']).round(3)
sig_dist.columns = ['Reports', 'Mean Score', 'Median Score']

strong_scores = signal_df[signal_df['signal_strength'] == 'strong']['score']
weak_scores = signal_df[signal_df['signal_strength'] == 'weak']['score']
u_sig, p_sig = mannwhitneyu(strong_scores, weak_scores, alternative='two-sided')
n1s, n2s = len(strong_scores), len(weak_scores)
rbc_sig = 1 - (2 * u_sig) / (n1s * n2s)

sig_sent = signal_df.groupby(['signal_strength', 'sentiment']).size().unstack(fill_value=0)
sig_sent_pct = sig_sent.div(sig_sent.sum(axis=1), axis=0) * 100
sig_sent_pct = sig_sent_pct.reindex(['strong', 'moderate', 'weak'])

fig, ax = plt.subplots(figsize=(8, 5))
bottom_arr = np.zeros(len(sig_sent_pct))
sent_colors = {'negative': '#e74c3c', 'mixed': '#95a5a6', 'positive': '#2ecc71', 'neutral': '#bdc3c7'}
for sent in ['negative', 'mixed', 'positive', 'neutral']:
    if sent in sig_sent_pct.columns:
        vals = sig_sent_pct[sent].values
        ax.bar(sig_sent_pct.index, vals, bottom=bottom_arr, label=sent.title(),
               color=sent_colors.get(sent, '#95a5a6'))
        for ii, (v, b) in enumerate(zip(vals, bottom_arr)):
            if v > 5:
                ax.text(ii, b + v/2, f'{v:.0f}%', ha='center', va='center', fontsize=10, fontweight='bold')
        bottom_arr = bottom_arr + vals

ax.set_ylabel('Percentage of Reports (%)', fontsize=12)
ax.set_xlabel('Signal Strength', fontsize=12)
ax.set_title('Sentiment Distribution by Signal Strength\n(SSRI/SNRI Reports Only)', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.set_ylim(0, 105)
fig.tight_layout()
plt.show()

display(HTML(sig_dist.to_html()))
if p_sig < 0.05:
    display(HTML(f"<p><b>Strong vs Weak signal:</b> Mann-Whitney U p={p_sig:.4f}, rank-biserial r={rbc_sig:.3f}. Strong-signal reports carry significantly more negative sentiment -- users who explicitly blame a drug express greater harm.</p>"))
else:
    display(HTML(f"<p><b>Strong vs Weak signal:</b> Mann-Whitney U p={p_sig:.4f}, rank-biserial r={rbc_sig:.3f}. No significant difference -- both strong and weak signals are overwhelmingly negative in this community.</p>"))


**What this means:** Signal strength provides some differentiation within the overwhelmingly negative landscape. Strong-signal reports -- where users explicitly blame the drug -- tend toward more uniform negativity. This makes intuitive sense: someone writing explicitly causal language is expressing greater certainty of harm than someone who merely mentions being on the drug.

### Sensitivity Check

Do the main findings hold if we restrict to strong-signal reports only? This filters out ambiguous mentions and keeps only cases where the user explicitly connected the drug to an outcome.

In [ ]:

strong_only = signal_df[signal_df['signal_strength'] == 'strong'].copy()
strong_user_drug = strong_only.groupby(['user_id', 'drug_merged']).agg(avg_score=('score', 'mean')).reset_index()
strong_user_drug['outcome'] = strong_user_drug['avg_score'].apply(classify_outcome)

strong_stats_list = []
for drug, grp in strong_user_drug.groupby('drug_merged'):
    n = len(grp)
    if n < 3:
        continue
    n_neg = (grp['outcome'] == 'negative').sum()
    strong_stats_list.append({'drug': drug, 'n_strong': n, 'neg_rate_strong': n_neg / n})

strong_compare = pd.DataFrame(strong_stats_list)
full_compare = drug_df[['drug', 'users', 'neg_rate']].rename(columns={'users': 'n_full', 'neg_rate': 'neg_rate_full'})
merged_compare = full_compare.merge(strong_compare, on='drug', how='inner').sort_values('n_full', ascending=False)

comp_display = merged_compare.copy()
comp_display['Drug'] = comp_display['drug'].str.title()
comp_display['All Reports'] = comp_display.apply(lambda r: f"{r['neg_rate_full']:.0%} (n={r['n_full']})", axis=1)
comp_display['Strong Only'] = comp_display.apply(lambda r: f"{r['neg_rate_strong']:.0%} (n={r['n_strong']})", axis=1)
comp_display['Shift'] = comp_display.apply(lambda r: f"{(r['neg_rate_strong'] - r['neg_rate_full'])*100:+.0f}pp", axis=1)

display(HTML("<h4>Sensitivity: Full Dataset vs Strong-Signal Only</h4>"))
display(HTML(comp_display[['Drug', 'All Reports', 'Strong Only', 'Shift']].to_html(index=False)))

max_shift = abs(merged_compare['neg_rate_strong'] - merged_compare['neg_rate_full']).max()
if max_shift < 0.15:
    display(HTML(f"<p><b>Sensitivity verdict:</b> Restricting to strong-signal reports does not meaningfully change the ranking or rates. The maximum shift is {max_shift*100:.0f} percentage points. The findings are robust to signal-strength filtering.</p>"))
else:
    display(HTML(f"<p><b>Sensitivity verdict:</b> Some drugs shift notably when restricted to strong signals (max shift: {max_shift*100:.0f}pp), suggesting signal strength matters for interpretation of those drugs.</p>"))


## Counterintuitive Findings Worth Investigating

In [ ]:

findings = []

# Vortioxetine: marketed as sexually-sparing
vort_users = user_drug_merged[user_drug_merged['drug_merged'] == 'vortioxetine']
vort_neg = (vort_users['outcome'] == 'negative').sum()
vort_n = len(vort_users)
if vort_n >= 5:
    findings.append({
        'finding': 'Vortioxetine (Trintellix): "sexually-sparing" SSRI with zero positive reports',
        'detail': f'Vortioxetine is marketed specifically for its lower sexual side effect profile compared to other SSRIs. Yet in this community, {vort_neg}/{vort_n} users ({vort_neg/vort_n:.0%}) report negative outcomes with zero positive reports. While the sample is small, the complete absence of any positive signal contradicts its clinical positioning.',
        'caveat': 'n=8. Selection bias likely applies: users whose vortioxetine caused PSSD may be more likely to post precisely because it was supposed to be safer, producing a sense of betrayal that drives engagement.'
    })

# Paroxetine underrepresented
par_n = len(user_drug_merged[user_drug_merged['drug_merged'] == 'paroxetine'])
sert_n_check = len(user_drug_merged[user_drug_merged['drug_merged'] == 'sertraline'])
if par_n < sert_n_check * 0.3:
    findings.append({
        'finding': 'Paroxetine: clinically worst SSRI for sexual side effects, yet underrepresented',
        'detail': f'Clinical literature consistently identifies paroxetine (Paxil) as the SSRI with the highest rate of sexual side effects and the most difficult discontinuation. Yet only {par_n} users report it here vs {sert_n_check} for sertraline. This likely reflects prescribing trends -- sertraline prescriptions far outnumber paroxetine in recent years -- but the gap is larger than prescribing volume alone would predict.',
        'caveat': 'Prescribing volume is a major confounder. Paroxetine prescribing has declined substantially since the 2000s, so fewer current patients are exposed.'
    })

# Lexapro brand vs escitalopram generic
lex_brand = user_drug[user_drug['canonical_name'] == 'lexapro']
esci_generic = user_drug[user_drug['canonical_name'] == 'escitalopram']
if len(lex_brand) >= 5 and len(esci_generic) >= 5:
    lex_neg_rate = (lex_brand['avg_score'].apply(classify_outcome) == 'negative').mean()
    esci_neg_rate = (esci_generic['avg_score'].apply(classify_outcome) == 'negative').mean()
    if abs(lex_neg_rate - esci_neg_rate) > 0.1:
        findings.append({
            'finding': 'Lexapro vs escitalopram: same molecule, different reporting patterns',
            'detail': f'Lexapro (brand, n={len(lex_brand)}) shows {lex_neg_rate:.0%} negative vs escitalopram (generic, n={len(esci_generic)}) at {esci_neg_rate:.0%}. These are the same molecule (escitalopram oxalate). The gap may reflect that brand-name users are more likely to specifically name and blame the drug, or that brand-name users were on the drug longer (brand loyalty correlating with duration).',
            'caveat': 'Pharmacologically identical compounds cannot have different side effect profiles. Any difference is a reporting or recall artifact.'
        })

if findings:
    for f in findings:
        display(HTML(f"""
        <div style='border-left:4px solid #e67e22; padding:10px 15px; margin:10px 0; background:#fef9e7;'>
        <p><b>{f['finding']}</b></p>
        <p>{f['detail']}</p>
        <p style='font-size:0.9em; color:#666;'><i>Caveat: {f['caveat']}</i></p>
        </div>
        """))
else:
    display(HTML("<p>All findings aligned with community consensus and clinical expectations.</p>"))


## What Patients Are Saying *(experimental -- under development)*

Quotes from r/PSSD users illustrating the harm profiles identified above. Each quote is from a user who reported negative outcomes for the named drug.

In [ ]:

import re

def clean_quote(text, max_words=38):
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    if len(words) > max_words:
        text = ' '.join(words[:max_words]) + '...'
    return text

quotes = []

# Sertraline - physical numbness
sert_q = pd.read_sql(
    "SELECT p.body_text, date(p.post_date, 'unixepoch') as dt FROM posts p "
    "WHERE p.user_id IN (SELECT DISTINCT tr.user_id FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id WHERE t.canonical_name = 'sertraline' "
    "AND tr.sentiment = 'negative' AND tr.signal_strength = 'strong') "
    "AND (p.body_text LIKE '%sertraline%' OR p.body_text LIKE '%zoloft%') "
    "AND (p.body_text LIKE '%numb%' OR p.body_text LIKE '%sensation%') "
    "AND LENGTH(p.body_text) > 80 "
    "AND p.body_text NOT LIKE '%Your post%' AND p.body_text NOT LIKE '%Please check%' "
    "LIMIT 5", conn)
for _, row in sert_q.iterrows():
    sents = [s.strip() for s in re.split(r'[.!?\n]+', row['body_text']) if len(s.strip()) > 20]
    for s in sents:
        if any(w in s.lower() for w in ['numb', 'sensation']):
            quotes.append(('Sertraline', 'Physical numbness', clean_quote(s), row['dt']))
            break
    if any(q[0] == 'Sertraline' for q in quotes):
        break

# Lexapro - rapid onset
lex_q = pd.read_sql(
    "SELECT p.body_text, date(p.post_date, 'unixepoch') as dt FROM posts p "
    "WHERE p.user_id IN (SELECT DISTINCT tr.user_id FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id WHERE t.canonical_name IN ('lexapro','escitalopram') "
    "AND tr.sentiment = 'negative') "
    "AND (p.body_text LIKE '%lexapro%' OR p.body_text LIKE '%escitalopram%') "
    "AND LENGTH(p.body_text) > 80 "
    "AND p.body_text NOT LIKE '%Your post%' AND p.body_text NOT LIKE '%Please check%' "
    "AND (p.body_text LIKE '%days%' OR p.body_text LIKE '%week%' OR p.body_text LIKE '%destroyed%') "
    "LIMIT 5", conn)
for _, row in lex_q.iterrows():
    sents = [s.strip() for s in re.split(r'[.!?\n]+', row['body_text']) if len(s.strip()) > 20]
    for s in sents:
        if any(w in s.lower() for w in ['lexapro', 'escitalopram']) and any(w in s.lower() for w in ['days', 'week', 'destroyed']):
            quotes.append(('Escitalopram/Lexapro', 'Rapid onset after short exposure', clean_quote(s), row['dt']))
            break
    if any(q[0] == 'Escitalopram/Lexapro' for q in quotes):
        break

# Paroxetine
par_q = pd.read_sql(
    "SELECT p.body_text, date(p.post_date, 'unixepoch') as dt FROM posts p "
    "WHERE p.user_id IN (SELECT DISTINCT tr.user_id FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id WHERE t.canonical_name = 'paroxetine') "
    "AND (p.body_text LIKE '%paroxetine%' OR p.body_text LIKE '%paxil%') "
    "AND LENGTH(p.body_text) > 80 AND p.body_text NOT LIKE '%Your post%' "
    "LIMIT 5", conn)
for _, row in par_q.iterrows():
    sents = [s.strip() for s in re.split(r'[.!?\n]+', row['body_text']) if len(s.strip()) > 20]
    for s in sents:
        if any(w in s.lower() for w in ['paroxetine', 'paxil', 'anhedonia', 'years', 'sexual']):
            quotes.append(('Paroxetine', 'Long-term harm', clean_quote(s), row['dt']))
            break
    if any(q[0] == 'Paroxetine' for q in quotes):
        break

# Duloxetine
dul_q = pd.read_sql(
    "SELECT p.body_text, date(p.post_date, 'unixepoch') as dt FROM posts p "
    "WHERE p.user_id IN (SELECT DISTINCT tr.user_id FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id WHERE t.canonical_name = 'duloxetine') "
    "AND (p.body_text LIKE '%duloxetine%' OR p.body_text LIKE '%cymbalta%') "
    "AND LENGTH(p.body_text) > 80 "
    "AND p.body_text NOT LIKE '%Your post%' AND p.body_text NOT LIKE '%Please check%' "
    "LIMIT 5", conn)
for _, row in dul_q.iterrows():
    sents = [s.strip() for s in re.split(r'[.!?\n]+', row['body_text']) if len(s.strip()) > 20]
    for s in sents:
        if any(w in s.lower() for w in ['duloxetine', 'cymbalta', 'pain', 'prescri']):
            quotes.append(('Duloxetine', 'Prescribed for non-psychiatric indication', clean_quote(s), row['dt']))
            break
    if any(q[0] == 'Duloxetine' for q in quotes):
        break

# Contradictory quote
unc_q = pd.read_sql(
    "SELECT p.body_text, date(p.post_date, 'unixepoch') as dt FROM posts p "
    "WHERE p.user_id IN (SELECT DISTINCT tr.user_id FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id "
    "WHERE t.canonical_name IN ('sertraline','fluoxetine','lexapro','escitalopram') "
    "AND (tr.sentiment = 'mixed' OR tr.sentiment = 'positive')) "
    "AND (p.body_text LIKE '%not sure%' OR p.body_text LIKE '%wonder%' OR p.body_text LIKE '%whether%') "
    "AND LENGTH(p.body_text) > 80 "
    "AND p.body_text NOT LIKE '%Your post%' AND p.body_text NOT LIKE '%Please check%' "
    "ORDER BY RANDOM() LIMIT 10", conn)
for _, row in unc_q.iterrows():
    sents = [s.strip() for s in re.split(r'[.!?\n]+', row['body_text']) if len(s.strip()) > 20]
    for s in sents:
        if any(w in s.lower() for w in ['not sure', 'wonder', 'whether', 'dealing']):
            quotes.append(('Mixed', 'Uncertainty about causation (contradicts dominant narrative)', clean_quote(s), row['dt']))
            break
    if any(q[0] == 'Mixed' for q in quotes):
        break

for drug, theme, quote, dt in quotes[:5]:
    display(HTML(f"""
    <div style='border-left:4px solid #3498db; padding:8px 15px; margin:8px 0; background:#f0f7fb;'>
    <p style='font-size:0.85em; color:#666; margin-bottom:4px;'><b>{drug}</b> -- {theme} ({dt})</p>
    <p style='font-style:italic; margin:0;'>"{quote}"</p>
    </div>
    """))


## Harm Risk Assessment by Drug

Since this analysis focuses on harm rather than treatment efficacy, the tiered framework is adapted: instead of "recommended treatments," we present a **harm severity ranking** based on negative outcome rates, sample size, and symptom breadth. Tiers: Strong evidence (n>=30, CI lower bound >80%), Moderate (n>=15, neg rate >70%), Preliminary (n<15).

In [ ]:

domain_summary = pd.DataFrame(results_list)
harm_tiers = []

for _, row in drug_df.iterrows():
    drug = row['drug']
    n = row['users']
    neg_rate = row['neg_rate']
    neg_lo = row['neg_ci_lo']

    domain_row = domain_summary[domain_summary['drug'] == drug]
    if len(domain_row) > 0:
        domains_above_30 = sum(1 for col in ['Sexual', 'Emotional', 'Cognitive']
                               if domain_row.iloc[0].get(col, 0) > 30)
    else:
        domains_above_30 = 0

    if n >= 30 and neg_lo > 0.8:
        tier = 'STRONG EVIDENCE OF HARM'
        tier_color = '#e74c3c'
    elif n >= 15 and neg_rate > 0.7:
        tier = 'MODERATE EVIDENCE OF HARM'
        tier_color = '#e67e22'
    elif n >= 4:
        tier = 'PRELIMINARY SIGNAL'
        tier_color = '#f1c40f'
    else:
        continue

    harm_tiers.append({
        'Drug': drug.title(), 'Tier': tier, 'tier_color': tier_color,
        'Users': n, 'Neg Rate': f"{neg_rate:.0%}",
        'CI (95%)': f"({neg_lo:.0%} - {row['neg_ci_hi']:.0%})",
        'Domains >30%': domains_above_30})

tiers_df = pd.DataFrame(harm_tiers)

for tier_name in ['STRONG EVIDENCE OF HARM', 'MODERATE EVIDENCE OF HARM', 'PRELIMINARY SIGNAL']:
    tier_subset = tiers_df[tiers_df['Tier'] == tier_name]
    if len(tier_subset) == 0:
        continue
    color = tier_subset.iloc[0]['tier_color']
    display(HTML(f"<h4 style='color:{color};'>{tier_name}</h4>"))
    display(HTML(tier_subset[['Drug', 'Users', 'Neg Rate', 'CI (95%)', 'Domains >30%']].to_html(index=False)))

# Visual: one chart per tier
tier_names_present = [t for t in ['STRONG EVIDENCE OF HARM', 'MODERATE EVIDENCE OF HARM', 'PRELIMINARY SIGNAL']
                      if len(tiers_df[tiers_df['Tier'] == t]) > 0]
fig, axes = plt.subplots(1, len(tier_names_present), figsize=(4 * len(tier_names_present), 5))
if len(tier_names_present) == 1:
    axes = [axes]

for idx, tier_name in enumerate(tier_names_present):
    tier_subset = tiers_df[tiers_df['Tier'] == tier_name].sort_values('Users', ascending=True)
    ax = axes[idx]
    color = tier_subset.iloc[0]['tier_color']
    neg_vals = [float(r.strip('%')) for r in tier_subset['Neg Rate']]
    bars = ax.barh(range(len(tier_subset)), neg_vals, color=color, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(tier_subset)))
    ax.set_yticklabels([f"{row['Drug']} (n={row['Users']})" for _, row in tier_subset.iterrows()], fontsize=10)
    ax.set_xlim(0, 115)
    ax.set_xlabel('Neg Rate (%)', fontsize=10)
    short_name = tier_name.replace('EVIDENCE OF ', '').title()
    ax.set_title(short_name, fontsize=11, fontweight='bold', color=color)
    for bar, val in zip(bars, neg_vals):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{val:.0f}%', va='center', fontsize=10)

fig.suptitle('PSSD Harm Risk Tiers', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()


## Conclusion

In [ ]:

conclusion_html = """
<div style='line-height:1.7; font-size:14px;'>
<p><b>Which SSRIs cause the worst PSSD?</b> Based on this data, <b>sertraline (Zoloft)</b> is the most-reported causative agent by a wide margin (39 users, 94.9% negative, zero positive reports). This likely reflects its position as the most commonly prescribed SSRI rather than uniquely higher toxicity, but the sheer volume of reports makes the signal unmistakable. Paroxetine, duloxetine, and venlafaxine show 100% negative rates but with samples too small (n=4-7) to distinguish from the 80-90% range.</p>

<p>Escitalopram/Lexapro, despite being the second-most-reported drug (29 merged users), carries a notably lower negative rate (approximately 76%) and is the only SSRI where a meaningful fraction of users report anything other than harm. Whether this reflects genuine pharmacological differences or simply different patient demographics and reporting patterns cannot be determined from this data.</p>

<p><b>What predicts severity?</b> The strongest predictor is <b>symptom domain breadth</b>. Duloxetine and citalopram users report dysfunction across sexual, emotional, and cognitive domains at much higher rates than sertraline or fluoxetine users, whose complaints cluster more narrowly around sexual symptoms. Paroxetine users show the highest sexual-symptom rate (85.7%), consistent with its pharmacological profile. <b>Polypharmacy</b> (multiple SSRI trials) does not significantly worsen outcomes beyond the floor effect. <b>Signal strength</b> serves as a modest severity proxy, with strong-signal reports showing more uniform negativity.</p>

<p>A patient asking "which SSRI is least likely to cause PSSD?" will not find a safe answer in this data -- all SSRIs carry substantial negative reporting. The data suggests that if PSSD risk must be weighed, escitalopram shows the least-dire profile in this community, but with wide uncertainty. The more important question for a prescriber may not be "which SSRI" but "whether an SSRI at all" -- a question this dataset cannot answer because it contains only people for whom SSRIs already went wrong. A clinician should note the vortioxetine signal: a drug marketed as sexually-sparing showing 0% positive reports warrants closer scrutiny, even at n=8.</p>
</div>
"""
display(HTML(conclusion_html))


## Research Limitations

In [ ]:

limitations_html = """
<div style='font-size:13px; line-height:1.6;'>
<ol>
<li><b>Selection bias:</b> r/PSSD members are self-selected -- they joined because an SSRI harmed them. This community cannot tell us about SSRI users who never developed PSSD. A drug with 95% negative here might have a population incidence well below 1%.</li>

<li><b>Reporting bias:</b> Users who feel strongly about their harm are more likely to post, and to post repeatedly. Sertraline's dominance may partly reflect its higher prescribing volume generating more affected users, not higher per-prescription risk.</li>

<li><b>Survivorship bias:</b> Users who recovered from PSSD may leave the community. Those who remain are disproportionately severe or long-duration cases, inflating negative rates for all drugs.</li>

<li><b>Recall bias:</b> Many users report on SSRI use from years or decades ago. Memory distortion, attribution errors, and retrospective severity inflation affect which drug gets blamed and how severely the experience is described.</li>

<li><b>Confounding:</b> Users who tried multiple SSRIs are inherently different from single-SSRI users -- they may have more treatment-resistant depression, longer psychiatric histories, or higher baseline vulnerability. Drug comparisons are confounded by indication.</li>

<li><b>No control group:</b> Without matched SSRI users who did NOT develop PSSD, we cannot calculate true risk ratios. The "negative rate" here is a within-community signal, not an incidence rate.</li>

<li><b>Sentiment vs. clinical severity:</b> Our text-mining pipeline classifies sentiment, not clinical severity. A user expressing intense emotional distress and one describing the same symptoms calmly receive different scores despite potentially identical clinical pictures.</li>

<li><b>Temporal snapshot:</b> One month of data (March-April 2026) captures a cross-section. Seasonal effects, viral posts, or community events could shift which drugs are discussed. Longer-term data would provide more stable estimates.</li>
</ol>
</div>
"""
display(HTML(limitations_html))


In [ ]:

display(HTML('<p style="font-size: 1.2em; font-weight: bold; font-style: italic;">These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</p>'))
